# Step 1: Import Libraries

In [1]:
import os
import pandas as pd
import numpy as np

from google.colab import drive

# Step 2: Load Image Records

Load the image records saved during data exploration, including image paths, class labels, image IDs, and patient IDs.

In [2]:
# Mount Google Drive
drive.mount("/content/drive")

# Set dataset path
csv_path = "/content/drive/MyDrive/COMP3710/project/dataset/processed/image_records.csv"

# Load CSV
df = pd.read_csv(
    csv_path,
    dtype={
        "image_id": str,
        "patient_id": str
    }
)

print("Total images:", len(df))
print("Missing patient IDs:", df["patient_id"].isna().sum())

print("\nFirst 5 records:")
print(df.head())

Mounted at /content/drive
Total images: 30521
Missing patient IDs: 0

First 5 records:
   split label        filename image_id  \
0  train    AD  218391_78.jpeg   218391   
1  train    AD  218391_79.jpeg   218391   
2  train    AD  218391_80.jpeg   218391   
3  train    AD  218391_81.jpeg   218391   
4  train    AD  218391_82.jpeg   218391   

                                                path  patient_id  
0  /content/drive/MyDrive/COMP3710/project/datase...  037_S_4001  
1  /content/drive/MyDrive/COMP3710/project/datase...  037_S_4001  
2  /content/drive/MyDrive/COMP3710/project/datase...  037_S_4001  
3  /content/drive/MyDrive/COMP3710/project/datase...  037_S_4001  
4  /content/drive/MyDrive/COMP3710/project/datase...  037_S_4001  


# Step 3: Check Original Patient Overlap

Identify unique patients in the original training and test sets and detect patient-level overlap.

In [3]:
# Get original train and test records
original_train = df[df["split"] == "train"]
original_test = df[df["split"] == "test"]

# Get unique patient IDs
train_subjects = set(original_train["patient_id"])
test_subjects = set(original_test["patient_id"])

# Find overlapping patients
overlap = train_subjects & test_subjects

print("Train patients:", len(train_subjects))
print("Test patients:", len(test_subjects))
print("Overlapping patients:", len(overlap))
print("Total unique patients:", len(train_subjects | test_subjects))

Train patients: 565
Test patients: 331
Overlapping patients: 216
Total unique patients: 680


# Step 4: Resolve Patient-Level Overlap

Assign overlapping patients to either the training or test set using a fixed random seed. The assignment is performed separately for AD and NC patients to help preserve class balance.

Images from overlapping patients are retained only in their assigned original partition.

In [4]:
# Set random seed for reproducibility
rng = np.random.default_rng(42)

# Build patient-to-label mapping
patient_labels = {}

for patient_id in df["patient_id"].unique():

    patient_records = df[df["patient_id"] == patient_id]

    labels = patient_records["label"].unique()

    if len(labels) != 1:
        raise ValueError(f"Conflicting labels for patient {patient_id}")

    patient_labels[patient_id] = labels[0]

# Separate overlapping patients by class
overlap_ad = []
overlap_nc = []

for patient_id in overlap:

    if patient_labels[patient_id] == "AD":
        overlap_ad.append(patient_id)

    elif patient_labels[patient_id] == "NC":
        overlap_nc.append(patient_id)

    else:
        raise ValueError(f"Unexpected label for patient {patient_id}")

# Sort before shuffling for reproducibility
overlap_ad = sorted(overlap_ad)
overlap_nc = sorted(overlap_nc)

rng.shuffle(overlap_ad)
rng.shuffle(overlap_nc)

# Assign approximately half to each split
train_overlap = set()
test_overlap = set()

for patients in [overlap_ad, overlap_nc]:

    midpoint = len(patients) // 2

    for patient_id in patients[:midpoint]:
        train_overlap.add(patient_id)

    for patient_id in patients[midpoint:]:
        test_overlap.add(patient_id)

print("Overlapping AD patients:", len(overlap_ad))
print("Overlapping NC patients:", len(overlap_nc))

print("\nAssigned to Train:", len(train_overlap))
print("Assigned to Test:", len(test_overlap))

Overlapping AD patients: 64
Overlapping NC patients: 152

Assigned to Train: 108
Assigned to Test: 108


In [5]:
# Keep patients that originally appeared in only one split
train_only = train_subjects - overlap
test_only = test_subjects - overlap

# Create final patient groups
final_train_subjects = train_only | train_overlap
final_test_subjects = test_only | test_overlap

# Keep image records in their assigned original split
train_df = original_train[
    original_train["patient_id"].isin(final_train_subjects)
].copy()

test_df = original_test[
    original_test["patient_id"].isin(final_test_subjects)
].copy()

print("Final Train images:", len(train_df))
print("Final Test images:", len(test_df))

print("\nFinal Train patients:", train_df["patient_id"].nunique())
print("Final Test patients:", test_df["patient_id"].nunique())

Final Train images: 17341
Final Test images: 6220

Final Train patients: 457
Final Test patients: 223


# Step 5: Verify Final Train/Test Splits

Verify that the final training and test sets contain no overlapping patients, no conflicting patient labels, and no missing patient IDs.

In [6]:
# Check patient overlap
final_overlap = (
    set(train_df["patient_id"])
    &
    set(test_df["patient_id"])
)

# Check missing patient IDs
missing_train = train_df["patient_id"].isna().sum()
missing_test = test_df["patient_id"].isna().sum()

# Check that both classes are present
print("Final Train class distribution:")
print(train_df["label"].value_counts())

print("\nFinal Test class distribution:")
print(test_df["label"].value_counts())

print("\nTrain/Test overlapping patients:", len(final_overlap))
print("Missing Train patient IDs:", missing_train)
print("Missing Test patient IDs:", missing_test)

# Final checks
assert len(final_overlap) == 0
assert missing_train == 0
assert missing_test == 0

assert set(train_df["label"]) == {"AD", "NC"}
assert set(test_df["label"]) == {"AD", "NC"}

assert len(train_df) > 0
assert len(test_df) > 0

print("\nPASS: Patient-independent Train/Test splits verified.")

Final Train class distribution:
label
NC    9161
AD    8180
Name: count, dtype: int64

Final Test class distribution:
label
AD    3440
NC    2780
Name: count, dtype: int64

Train/Test overlapping patients: 0
Missing Train patient IDs: 0
Missing Test patient IDs: 0

PASS: Patient-independent Train/Test splits verified.


In [7]:
original_patients = set(df["patient_id"])

final_patients = (
    set(train_df["patient_id"])
    |
    set(test_df["patient_id"])
)

print("Original unique patients:", len(original_patients))
print("Final unique patients:", len(final_patients))

assert original_patients == final_patients

print("PASS: All original patients are represented.")

Original unique patients: 680
Final unique patients: 680
PASS: All original patients are represented.


# Step 6: Save Final Data Splits

Save the leakage-free training and test image records as CSV files for consistent CNN and GFNet experiments.

In [8]:
# Set output folder
output_folder = "/content/drive/MyDrive/COMP3710/project/dataset/splits"

os.makedirs(output_folder, exist_ok=True)

# Set output paths
train_path = os.path.join(output_folder, "train.csv")
test_path = os.path.join(output_folder, "test.csv")

# Save CSV files
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved Train to:", train_path)
print("Saved Test to:", test_path)

print("\nTrain images:", len(train_df))
print("Test images:", len(test_df))

Saved Train to: /content/drive/MyDrive/COMP3710/project/dataset/splits/train.csv
Saved Test to: /content/drive/MyDrive/COMP3710/project/dataset/splits/test.csv

Train images: 17341
Test images: 6220


In [9]:
# Reload saved CSV files
saved_train = pd.read_csv(
    train_path,
    dtype={"image_id": str, "patient_id": str}
)

saved_test = pd.read_csv(
    test_path,
    dtype={"image_id": str, "patient_id": str}
)

# Verify saved records
assert len(saved_train) == len(train_df)
assert len(saved_test) == len(test_df)

# Verify no patient overlap
assert set(saved_train["patient_id"]).isdisjoint(
    set(saved_test["patient_id"])
)

print("PASS: Saved Train and Test CSV files verified.")

PASS: Saved Train and Test CSV files verified.
